# 变长注意力优化：从 Sequence Packing 到 TND Attention

第 5 章介绍了 Qwen3 的 SDPA 配置路径、Attention backend dispatch 和 profiler 证据。本章把这些基础应用到 SFT 的变长样本：先理解 packing 为什么会让不同样本互相看见，再让 Attention 只读取同一样本，最后用端到端 profiling 验证收益。

本节只负责章节导航；问题细节从下一节开始。

---


## 教程进度回顾

| 章节 | 内容 | 状态 |
|---|---|---|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| 第 4 章 | 融合算子优化 + Profiling | ✅ 已完成 |
| 第 5 章 | Attention 公式、算子与 TorchTitan dispatch | ✅ 已完成 |
| **第 6 章** | **Sequence Packing、Block-Causal Mask、Varlen 与端到端效率** | ← 当前 |

---


## 本章目标

学完本章，你应该能够：

- 解释 non-greedy 与 greedy sequence packing 的差异；
- 识别 packed SFT 中普通 causal SDPA 的跨文档语义问题；
- 沿 TorchTitan `model_spec` 路径接入 block-causal 与 Varlen attention；
- 用有效 token/s、原始样本/s、step time 和 device trace 解释端到端收益。

---


## 前置条件

- 完成第 3 章，能够运行 Wordle SFT recipe；
- 完成第 4 章，理解 correctness → trace → 重复计时的 profiling 流程；
- 完成第 5 章，熟悉 SDPA、causal mask、Q/K/V shape 与 TorchTitan inner attention；
- 当前环境能够读取 TorchTitan 与 TorchTitan-NPU 源码。

---


## 本章结构

| Notebook | 内容 |
|---|---|
| [06.01](06.01_chapter_intro.ipynb) | 章节介绍（本节） |
| [06.02](06.02_problem_statement.ipynb) | Packed SFT 的 Attention 问题与四条对照路线 |
| [06.03](06.03_dataloader_and_packing.ipynb) | Non-greedy/greedy packing、sample boundaries、positions 与 labels |
| [06.04](06.04_dataloader_profiling.ipynb) | DataLoader 变化后的 packing、padding 与安全 SDPA baseline profiling |
| [06.05](06.05_from_sdpa_to_varlen.ipynb) | Causal → block-causal → Varlen，以及 `model_spec` 配置路径 |
| [06.06](06.06_end_to_end_profiling.ipynb) | Backend 变化后的受控 Attention 对比、端到端 profiling 与收益归因 |
| [06.07](06.07_chapter_practice.ipynb) | 按 06.01–06.06 分组的判断题与选择题 |

---


## 本章产物与验收

完成后应能交付：

1. 一份明确区分完整训练样本与样本内部聊天消息的 DataLoader 说明；
2. 一条可检查的 `sft_qwen3_1_7b_wordle_tnd()` model-spec wrapper；
3. C/D 两条正确性一致的受控 attention 对照；
4. B/D 两条生产路线的端到端吞吐与 trace 结果。

当前实现已经包含 non-greedy/greedy 分流、dense block-causal SDPA 和 NPU VarLen 路线。DataLoader 让每条 packed sample 的位置编号从 0 重新开始，trainer 据此恢复样本起点；本章只描述这条当前可运行路径。


## 练习

1. （判断题）第 6 章只需要把一个 Attention kernel 换成另一个，DataLoader 如何表达样本边界不影响正确性。

2. （单选题）本章验证 packed SFT 的完整链路应按什么顺序展开？
    A. Greedy packing → 位置编号重置 → 边界 metadata → block-causal/VarLen → correctness 与 trace
    B. 提高 GBS → 只看 loss → 推断 kernel
    C. 修改 tokenizer → 删除 padding → 跳过 profiler
    D. 只比较一个 Attention kernel 的耗时

3. （判断题）两条路线处理相同数量的容器位置，就能断定它们每步处理的原始样本数和 supervised token 数相同。

4. （多选题）第 6 章中哪些工作可以在 CPU/离线环境完成？
    A. 阅读 packing、positions 与 labels 的数据契约
    B. 计算 dense 与 VarLen 的理论 Attention pair 数
    C. 检查 model-spec 与配置路径
    D. 测量 Ascend NPU 上真实 kernel 与训练 step 时间

In [ ]:
!cat ./answer/06.01_answer.txt
